# Local Nutrition CallBot Pipeline + RAGAS
Pipeline notebook to run everything locally: start Qdrant, restore snapshot, launch brain server, batch-evaluate QA, and score with RAGAS (Gemini API or Vertex).

In [1]:
# CELL 1: Install dependencies (run once)
import sys, subprocess
pkgs = [
    "ragas>=0.2.0",
    "langchain-google-genai",
    "google-genai>=1.0.0",
    "qdrant-client>=1.7.0",
    "pandas",
    "matplotlib",
    "seaborn",
    "requests",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# CELL 2: Configure paths, secrets, and sanity checks
import os, json, pathlib
BASE_DIR = pathlib.Path(".").resolve()
DATA_DIR = BASE_DIR / "evaluation"
QA_PATH = os.environ.get("QA_PATH", str(DATA_DIR / "thucuc_qa.jsonl"))
SNAPSHOT_PATH = os.environ.get("SNAPSHOT_PATH", "./nutrition_articles.snapshot")
COLLECTION = os.environ.get("QDRANT_COLLECTION", "nutrition_articles")
EVAL_RESULTS = os.environ.get("EVAL_RESULTS", "./eval_results.jsonl")
RAGAS_CSV = os.environ.get("RAGAS_CSV", "./ragas_results.csv")
MODEL = os.environ.get("MODEL", "gemini-2.5-flash")
USE_VERTEX = os.environ.get("USE_VERTEX", "false").lower() == "true"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
VERTEX_PROJECT = os.environ.get("VERTEX_PROJECT", "")
VERTEX_LOCATION = os.environ.get("VERTEX_LOCATION", "us-central1")
print("Working dir :", BASE_DIR)
print("QA_PATH     :", QA_PATH)
print("SNAPSHOT    :", SNAPSHOT_PATH)
print("EVAL_RESULTS:", EVAL_RESULTS)
print("RAGAS_CSV   :", RAGAS_CSV)
print("MODEL       :", MODEL)
print("USE_VERTEX  :", USE_VERTEX)
if not pathlib.Path(QA_PATH).exists():
    raise FileNotFoundError(f"QA_PATH missing: {QA_PATH}")
if USE_VERTEX and not VERTEX_PROJECT:
    print("USE_VERTEX=True but VERTEX_PROJECT missing → will fallback to GEMINI_API_KEY")
if not GEMINI_API_KEY and not USE_VERTEX:
    print("WARNING: GEMINI_API_KEY is empty; set env before running brain or RAGAS")

In [ ]:
# CELL 3: Start local Qdrant service
import subprocess, time, requests, shutil
QDRANT_BIN = "./qdrant"
if not pathlib.Path(QDRANT_BIN).exists():
    url = "https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz"
    print("Downloading Qdrant binary...")
    import tarfile, io
    import urllib.request
    with urllib.request.urlopen(url) as resp:
        data = resp.read()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
        tar.extractall(path=BASE_DIR)
    print("Downloaded Qdrant")
# Kill any existing instance
subprocess.run(["pkill", "-f", "qdrant"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
qdrant_proc = subprocess.Popen([QDRANT_BIN], stdout=open("qdrant.log", "w"), stderr=subprocess.STDOUT)
print(f"Qdrant PID={qdrant_proc.pid}, waiting for health...")
for i in range(30):
    try:
        r = requests.get("http://localhost:6333/healthz", timeout=1)
        if r.status_code == 200:
            print("Qdrant ready")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Qdrant did not become ready; check qdrant.log")

In [ ]:
# CELL 4: Restore collection from snapshot
import requests
if not pathlib.Path(SNAPSHOT_PATH).exists():
    raise FileNotFoundError(f"Snapshot not found: {SNAPSHOT_PATH}")
print("Uploading snapshot to Qdrant...")
resp = requests.post(
    f"http://localhost:6333/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
    files={"snapshot": open(SNAPSHOT_PATH, "rb")},
    timeout=300,
 )
print("Status:", resp.status_code)
if resp.status_code != 200:
    raise RuntimeError(resp.text)
info = requests.get(f"http://localhost:6333/collections/{COLLECTION}").json()
print("Collection ready: vectors=", info.get("result", {}).get("vectors_count"))

In [ ]:
# CELL 5: Launch brain server (Gemini backend)
import subprocess, sys, time, os
os.environ.setdefault("GEMINI_API_KEY", GEMINI_API_KEY)
os.environ.setdefault("QDRANT_URL", "http://localhost:6333")
os.environ.setdefault("QDRANT_API_KEY", "")
os.environ.setdefault("LLM_BACKEND", "gemini")
os.environ.setdefault("LLM_MODEL", MODEL)
BRAIN_DIR = BASE_DIR / "nutrition-callbot" / "brain"
if not BRAIN_DIR.exists():
    raise FileNotFoundError(f"Brain dir not found: {BRAIN_DIR}")
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_DIR) + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(["pkill", "-f", "brain.server"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
brain_proc = subprocess.Popen([sys.executable, "-m", "brain.server"], cwd=BRAIN_DIR, env=env, stdout=open("brain.log", "w"), stderr=subprocess.STDOUT)
print(f"Brain PID={brain_proc.pid}, waiting for health...")
for i in range(60):
    try:
        r = requests.get("http://localhost:50052/health", timeout=2)
        if r.status_code == 200:
            print("Brain ready")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print("Brain not ready; check brain.log")

In [ ]:
# CELL 6: Batch evaluation and save JSONL
import pandas as pd, numpy as np
import time
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "80"))
SLEEP_BETWEEN_REQS = float(os.environ.get("SLEEP_BETWEEN_REQS", "0.4"))
records = []
with open(QA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))
if EVAL_LIMIT:
    records = records[:EVAL_LIMIT]
print(f"Evaluating {len(records)} questions...")
results = []
for i, rec in enumerate(records):
    t0 = time.time()
    try:
        resp = requests.post(
            "http://localhost:50052/think/stream",
            json={"query": rec["question"], "session_id": f"eval_{i}"},
            stream=True, timeout=60,
        )
        answer = ""
        contexts = []
        timing = {}
        ttft_s = None
        for line in resp.iter_lines():
            if not line:
                continue
            d = json.loads(line.decode())
            if d.get("text") and not d.get("is_final"):
                if ttft_s is None:
                    ttft_s = time.time() - t0
                answer += d["text"]
            if d.get("is_final"):
                contexts = d.get("contexts", [])
                timing = d.get("timing", {})
        results.append({
            "question": rec["question"],
            "answer": answer,
            "ground_truth": rec.get("answer"),
            "contexts": contexts,
            "timing": timing,
            "ttft_s": round(ttft_s or 0, 3),
            "latency_s": round(time.time() - t0, 2),
        })
    except Exception as e:
        results.append({"question": rec.get("question"), "error": str(e)})
    finally:
        time.sleep(SLEEP_BETWEEN_REQS)
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(records)} done")
with open(EVAL_RESULTS, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
errors = sum(1 for r in results if "error" in r)
avg_lat = sum(r.get("latency_s", 0) for r in results if "error" not in r) / max(1, len(results) - errors)
print(f"Done: {len(results)} | Errors: {errors} | Avg latency: {avg_lat:.1f}s")
print(f"Saved: {EVAL_RESULTS}")

In [ ]:
# CELL 7: One-cell RAGAS evaluation (Gemini or Vertex)
from ragas import evaluate, EvaluationDataset, SingleTurnSample
try:
    from ragas.metrics.collections import (
        ContextPrecision,
        ContextRecall,
        Faithfulness,
        AnswerRelevancy,
        FactualCorrectness,
    )
    LLMContextPrecisionWithReference = ContextPrecision
    ResponseRelevancy = AnswerRelevancy
    METRIC_SRC = "collections"
except Exception:
    try:
        from ragas.metrics import (
            LLMContextPrecisionWithReference,
            ContextRecall,
            Faithfulness,
            ResponseRelevancy,
            FactualCorrectness,
        )
        METRIC_SRC = "legacy"
    except Exception as e:
        raise ImportError("Cannot import ragas metrics; install ragas") from e
from ragas.llms import LangchainLLMWrapper
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ModuleNotFoundError:
    raise ModuleNotFoundError("Missing langchain-google-genai; pip install langchain-google-genai google-genai ragas")
# Load eval results
records = []
with open(EVAL_RESULTS) as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))
ok = [r for r in records if "error" not in r and r.get("contexts")]
print(f"Valid records: {len(ok)}/{len(records)}")
EVAL_N = None if os.environ.get("EVAL_N") in (None, "") else int(os.environ.get("EVAL_N"))
subset = ok if EVAL_N is None else ok[:EVAL_N]
print(f"Scoring {len(subset)} samples")
samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        reference=r.get("ground_truth"),
        retrieved_contexts=r["contexts"],
    )
    for r in subset
]
dataset = EvaluationDataset(samples=samples)
if USE_VERTEX and VERTEX_PROJECT:
    llm_raw = ChatGoogleGenerativeAI(model=MODEL, project=VERTEX_PROJECT, location=VERTEX_LOCATION)
else:
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY missing; set env")
    llm_raw = ChatGoogleGenerativeAI(model=MODEL, google_api_key=GEMINI_API_KEY)
llm = LangchainLLMWrapper(llm_raw)
def build_metrics():
    if METRIC_SRC == "collections":
        return [
            LLMContextPrecisionWithReference(llm=llm),
            ContextRecall(llm=llm),
            Faithfulness(llm=llm),
            ResponseRelevancy(llm=llm),
            FactualCorrectness(llm=llm),
        ]
    return [
        LLMContextPrecisionWithReference(),
        ContextRecall(),
        Faithfulness(),
        ResponseRelevancy(),
        FactualCorrectness(),
    ]
metrics = build_metrics()
result = evaluate(dataset=dataset, metrics=metrics, llm=llm)
print(result)

In [ ]:
# CELL 8: Export metrics to CSV and quick stats
import pandas as pd, numpy as np
df = result.to_pandas()
df.to_csv(RAGAS_CSV, index=False)
print(f"Saved: {RAGAS_CSV}")
print("Mean scores:")
print(df.mean(numeric_only=True))
if subset and isinstance(subset[0], dict):
    df_lat = pd.DataFrame(subset)
    if "latency_s" in df_lat:
        vals = df_lat["latency_s"].dropna().astype(float)
        if not vals.empty:
            for p in [50, 90, 95, 99]:
                print(f"Latency p{p}: {np.percentile(vals, p):.1f}s")